# Laboratorio 8 - DuckDB

Se consultan directamente los archivos Parquet y se revisan los resultados de las tres etapas. Las interpretaciones completas est?n en docs/informe.md. Los filtros est?n registrados en scripts/common.py.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd()
if root.name == 'notebooks':
    root = root.parent
sys.path.insert(0, str(root / 'scripts'))
from common import connect, files, views, sql
import pandas as pd
from IPython.display import display, Image
con = connect()
views(con, files([2024, 2025, 2026]))

## Archivos, registros, columnas y tipos

Se usa la vista directa de Parquet. La cantidad de archivos y registros se obtiene por tipo y a?o.

In [ ]:
display(con.execute(sql('coverage')).fetchdf())
display(pd.read_csv(root / 'docs/results/2024_2025_2026/schema.csv'))

## Muestra y calidad

Se revisan diez registros. La muestra ordenada no representa todos los viajes. Los problemas de calidad pueden superponerse.

In [ ]:
display(con.execute('SELECT * FROM trips_raw ORDER BY source_file, pickup LIMIT 10').fetchdf())
display(con.execute(sql('quality')).fetchdf())

## An?lisis exploratorio

Se revisan viajes por mes, horarios, pagos, distribuciones y zonas. Los resultados se calculan sobre trips_clean.

In [ ]:
for name in ['monthly', 'hourly', 'payments', 'distribution', 'zones']:
    print(name)
    display(pd.read_csv(root / 'docs/results/2024_2025_2026' / f'{name}.csv'))

## Incorporaci?n incremental

Se comparan los resultados registrados despu?s de cada ampliaci?n.

In [ ]:
for stage in ['2026', '2024_2026', '2024_2025_2026']:
    print(stage)
    display(pd.read_csv(root / 'docs/results' / stage / 'coverage.csv'))

## Comparaci?n de los mismos meses

Se evita comparar ocho meses de 2026 con doce de otro a?o. Se conserva NULL cuando el recargo de congesti?n no est? disponible.

In [ ]:
display(pd.read_csv(root / 'docs/results/2024_2025_2026/comparable.csv'))

## Benchmark

Se muestran tres repeticiones con cach? caliente para 2, 12 y todos los archivos de 2024 y 2026. El costo de materializaci?n se muestra por separado.

In [ ]:
display(pd.read_csv(root / 'docs/results/benchmark_summary.csv'))
display(pd.read_csv(root / 'docs/results/materialization.csv'))

## Tablero

Se generaron seis indicadores en Metabase. Esta imagen permite revisar las visualizaciones en conjunto.

In [ ]:
display(Image(filename=str(root / 'docs/dashboard/dashboard.png')))
con.close()